In [1]:
!pip install tensorflow  

In [7]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from glob import glob
from PIL import Image

# --- CONFIGURATION ---
data_root = r"C:\Users\ashut\Desktop\ElementaryCQT"
image_size = (200, 200)
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE

# --- STEP 1: Load All Images and Labels ---
def load_data():
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(data_root))
    class_map = {name: idx for idx, name in enumerate(class_names)}

    for shape_class in class_names:
        shape_dir = os.path.join(data_root, shape_class)
        for subtype in os.listdir(shape_dir):
            subtype_dir = os.path.join(shape_dir, subtype)
            if os.path.isdir(subtype_dir):
                for img_file in glob(f"{subtype_dir}/*.png") + glob(f"{subtype_dir}/*.jpg"):
                    image_paths.append(img_file)
                    labels.append(class_map[shape_class])

    return image_paths, labels, class_names

# --- STEP 2: Preprocess Images ---
def preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])  # Set shape manually
    image = tf.image.rgb_to_grayscale(image)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# --- STEP 3: Create Dataset Splits ---
image_paths, labels, class_names = load_data()
num_classes = len(class_names)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

def make_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)

print(f"Dataset sizes — Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

# --- STEP 4: Build Geo-CNN Model ---
class EdgeLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.edge_conv = tf.keras.layers.Conv2D(8, kernel_size=3, padding='same', activation='relu')

    def call(self, x):
        return self.edge_conv(x)

class ShapeAttention(tf.keras.layers.Layer):
    def call(self, x):
        # x: (B, H, W, C)
        attn = tf.reduce_mean(x, axis=-1, keepdims=True)  # (B, H, W, 1)
        flat_attn = tf.reshape(attn, [tf.shape(x)[0], -1])  # (B, H*W)
        norm_attn = tf.nn.softmax(flat_attn, axis=-1)  # attention over spatial locations
        norm_attn = tf.reshape(norm_attn, tf.shape(attn))  # back to (B, H, W, 1)
        return x * norm_attn  # apply attention mask to original input



class GeometricEmbedding(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.pool = tf.keras.layers.GlobalAveragePooling2D()
        self.dense = tf.keras.layers.Dense(32, activation='relu')

    def call(self, x):
        return self.dense(self.pool(x))

def build_geo_cnn(input_shape=(200, 200, 1), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)
    x = EdgeLayer()(inputs)
    x = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = ShapeAttention()(x)
    x = GeometricEmbedding()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)

# --- STEP 5: Compile and Train ---
model = build_geo_cnn(num_classes=num_classes)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_ds, validation_data=val_ds, epochs=10)

# --- STEP 6: Evaluate on Test Set ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"✅ Test Accuracy: {test_acc:.4f}")


Dataset sizes — Train: 304000, Val: 38000, Test: 38000
Epoch 1/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 984s 103ms/step - accuracy: 0.2809 - loss: 1.8168 - val_accuracy: 0.5465 - val_loss: 1.1388
Epoch 2/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 919s 97ms/step - accuracy: 0.6588 - loss: 0.8554 - val_accuracy: 0.7423 - val_loss: 0.6115
Epoch 3/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 915s 96ms/step - accuracy: 0.7649 - loss: 0.5801 - val_accuracy: 0.7542 - val_loss: 0.6221
Epoch 4/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 954s 100ms/step - accuracy: 0.7913 - loss: 0.5052 - val_accuracy: 0.8083 - val_loss: 0.4585
Epoch 5/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 915s 96ms/step - accuracy: 0.8084 - loss: 0.4599 - val_accuracy: 0.8201 - val_loss: 0.4294
Epoch 6/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 994s 105ms/step - accuracy: 0.8199 - loss: 0.4277 - val_accuracy: 0.8171 - val_loss: 0.4345
Epoch 7/10
9500/9500 ━━━━━━━━━━━━━━━━━━━━ 950s 100ms/step - accuracy: 0.8325 - loss: 0.3996 - val_accuracy: 0.8563 - val_loss: 0.3531
Epoch 8/10

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import os
import shutil

# --- STEP 1: Data Preparation ---
# Assumes you have a dataset structure with folders for each shape class
main_dir =  r"C:\Users\ashut\Desktop\ElementaryCQT"  # Update this to the path of your dataset
train_dir = os.path.join(main_dir, 'train')
val_dir = os.path.join(main_dir, 'val')
test_dir = os.path.join(main_dir, 'test')

# Helper function to split dataset
def split_dataset(source_dir, dest_dir_train, dest_dir_val, dest_dir_test, val_size=0.1, test_size=0.1):
    for shape_class in os.listdir(source_dir):
        shape_class_path = os.path.join(source_dir, shape_class)
        if os.path.isdir(shape_class_path):
            images = os.listdir(shape_class_path)
            train_imgs, test_imgs = train_test_split(images, test_size=test_size)
            train_imgs, val_imgs = train_test_split(train_imgs, test_size=val_size)

            # Create subdirectories for each class in train, val, and test dirs
            os.makedirs(os.path.join(dest_dir_train, shape_class), exist_ok=True)
            os.makedirs(os.path.join(dest_dir_val, shape_class), exist_ok=True)
            os.makedirs(os.path.join(dest_dir_test, shape_class), exist_ok=True)

            # Copy images into respective directories
            for img in train_imgs:
                shutil.copy(os.path.join(shape_class_path, img), os.path.join(dest_dir_train, shape_class, img))
            for img in val_imgs:
                shutil.copy(os.path.join(shape_class_path, img), os.path.join(dest_dir_val, shape_class, img))
            for img in test_imgs:
                shutil.copy(os.path.join(shape_class_path, img), os.path.join(dest_dir_test, shape_class, img))

# Split the dataset into train, validation, and test sets
split_dataset(main_dir, train_dir, val_dir, test_dir, val_size=0.1, test_size=0.1)

# --- STEP 2: Data Augmentation and Preprocessing ---
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_ds = train_datagen.flow_from_directory(
    train_dir,
    target_size=(200, 200),
    batch_size=32,
    class_mode='sparse')

val_ds = val_datagen.flow_from_directory(
    val_dir,
    target_size=(200, 200),
    batch_size=32,
    class_mode='sparse')

test_ds = test_datagen.flow_from_directory(
    test_dir,
    target_size=(200, 200),
    batch_size=32,
    class_mode='sparse')

# --- STEP 3: CNN Model Architecture ---
def build_geo_cnn(num_classes):
    inputs = layers.Input(shape=(200, 200, 3))
    
    # Convolutional layers with batch normalization and dropout
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.4)(x)

    # Flatten and fully connected layers
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)

# --- STEP 4: Model Compilation ---
model = build_geo_cnn(num_classes=len(train_ds.class_indices))  # Automatically gets the number of classes
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# --- STEP 5: Train the Model ---
history = model.fit(train_ds, validation_data=val_ds, epochs=20)

# --- STEP 6: Evaluate on Test Set ---
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_acc}")

# --- STEP 7: Save the Model ---
model.save('geometric_cnn_model.h5')

# --- STEP 8: Optionally, Visualize Training History ---
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()
